# Chapter 6 — Top-down Ontology Development
### Notebook 3 · Exercises

*Book reference: Section 6.3*

The book's exercises, executable. Assertions are the marking scheme.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch06_toolkit as ch6
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

### Exercise R1 — Classify six part-whole statements

For each statement, give the relation and say whether it is parthood.

In [3]:
statements = [
    ('A branch is part of a tree.', 'physical-object', 'physical-object'),
    ('The sugar is part of the syrup.', 'amount-of-matter', 'amount-of-matter'),
    ('The bronze is part of the sculpture.', 'amount-of-matter', 'physical-object'),
    ('A player is part of a team.', 'physical-object', 'collection'),
    ('The referee is part of the match.', 'physical-object', 'process'),
    ('Kneading is part of baking.', 'process', 'process'),
]
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [4]:
statements = [
    ('A branch is part of a tree.', 'physical-object', 'physical-object'),
    ('The sugar is part of the syrup.', 'amount-of-matter', 'amount-of-matter'),
    ('The bronze is part of the sculpture.', 'amount-of-matter', 'physical-object'),
    ('A player is part of a team.', 'physical-object', 'collection'),
    ('The referee is part of the match.', 'physical-object', 'process'),
    ('Kneading is part of baking.', 'process', 'process'),
]
rows = []
for text, part, whole in statements:
    r = ch6.relation_by_id(ch6.classify_partwhole(part, whole))
    rows.append({'statement': text, 'relation': r.name,
                 'parthood': r.parthood, 'transitive': r.transitive})
import pandas as pd; print(pd.DataFrame(rows).to_string(index=False))
expected = ['component of', 'sub-quantity of', 'constituted of',
            'member of', 'participates in', 'involved in']
assert [r['relation'] for r in rows] == expected
print('\nSix statements, six different relations, identical English. Two of the\n'
      'six are not parthood at all.')

                           statement        relation  parthood  transitive
         A branch is part of a tree.    component of      True       False
     The sugar is part of the syrup. sub-quantity of      True        True
The bronze is part of the sculpture.  constituted of     False       False
         A player is part of a team.       member of      True       False
   The referee is part of the match. participates in     False       False
         Kneading is part of baking.     involved in      True        True

Six statements, six different relations, identical English. Two of the
six are not parthood at all.


### Exercise R2 — Decide four chaining questions

For each pair of relations, say whether the chain is valid and why.

In [5]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [6]:
pairs = [('component-of', 'component-of'),
         ('sub-quantity-of', 'sub-quantity-of'),
         ('member-of', 'member-of'),
         ('participates-in', 'involved-in')]
for first, second in pairs:
    r = ch6.can_chain(first, second)
    print(f'{first:18s} + {second:18s} -> {str(r["valid"]):5s}  {r["reason"]}')
assert not ch6.can_chain('component-of', 'component-of')['valid']
assert ch6.can_chain('sub-quantity-of', 'sub-quantity-of')['valid']
assert not ch6.can_chain('member-of', 'member-of')['valid']
print('\nThe first is the surprise: componenthood is genuine parthood and still\n'
      'not transitive. A finger is a component of a hand and a hand of a body,\n'
      'but whether a finger is a component OF THE BODY depends on how you\n'
      'individuate components -- which is exactly why the relation is not\n'
      'declared transitive.')

component-of       + component-of       -> False  'component of' is parthood but is not transitive
sub-quantity-of    + sub-quantity-of    -> True   'sub-quantity of' is parthood and transitive
member-of          + member-of          -> False  'member of' is parthood but is not transitive
participates-in    + involved-in        -> False  'participates in' is not genuine parthood, so nothing about parthood follows from it

The first is the surprise: componenthood is genuine parthood and still
not transitive. A finger is a component of a hand and a hand of a body,
but whether a finger is a component OF THE BODY depends on how you
individuate components -- which is exactly why the relation is not
declared transitive.


### Exercise R3 — Choose between DOLCE and BFO for a project

A project must represent musical works, performances, and the scores they are written on. Say which foundational ontology fits and why, using the comparison table.

In [7]:
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [8]:
import pandas as pd
print(pd.DataFrame(ch6.BFO_COMPARISON).to_string(index=False))
needs = {
    'a musical work (abstract)': 'abstract',
    'a performance (happens)': 'process',
    'a score (physical copy)': 'physical-object',
}
for label, category in needs.items():
    print(f'  {label:28s} -> DOLCE {category}')
bfo_gap = [row for row in ch6.BFO_COMPARISON if 'none' in row['bfo']]
print('\nBFO has no category for:', [row['dolce'] for row in bfo_gap])
assert bfo_gap and bfo_gap[0]['dolce'] == 'Abstract Entity'
print('\nA musical work is not identical to any performance of it, nor to any\n'
      'printed score -- it is abstract. BFO is realist and excludes abstracta,\n'
      'so this project needs DOLCE (or must re-model works as something else,\n'
      'which is a real cost and should be a conscious decision).')

           dolce                                                 bfo                                           note
 Physical Object                            Material Entity (Object)                                  closest match
Amount of Matter        Material Entity (Object Aggregate / portion) BFO has no dedicated amount-of-matter category
         Feature Immaterial Entity (Site) / Continuant Fiat Boundary                         holes are sites in BFO
         Quality         Specifically Dependent Continuant (Quality)                                    close match
           Event                   Process (with a process boundary)                BFO folds events into processes
         Process                                             Process                                    close match
 Abstract Entity                             (none - BFO is realist)            BFO deliberately excludes abstracta
  a musical work (abstract)    -> DOLCE abstract
  a performance (happen

## Where this leaves you

You can align a class by interrogation, name the relation a "part of" statement really expresses, and say when a chain of them is sound. Notebook 4 hands both jobs to an agent — and asks an optimiser to *derive* the alignment questions rather than be told them.